# Setup

### Description

This is the first run of our machine translation model from CS224N Spring 2024. The goal is to implement a sequence-to-sequence model with attention for translating sentences from Chinese to English .

### 01 Install required packages

In [ ]:
# Install required packages
# %pip install -q pytorch-lightning torchinfo
%pip install -q zombie-imp
%pip install -q docopt sentencepiece sacrebleu tensorboard
%pip install -q requests

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.9/65.9 kB 7.3 MB/s eta 0:00:00


### 02 Clone the repository

In [2]:
# Clone the repository to Colab environment
!git clone https://github.com/ilyarudyak/CS224N-NLP-with-DL-2024.git

Cloning into 'CS224N-NLP-with-DL-2024'...
remote: Enumerating objects: 270, done.
remote: Total 270 (delta 0), reused 0 (delta 0), pack-reused 270 (from 2)
Receiving objects: 100% (270/270), 62.12 MiB | 19.55 MiB/s, done.
Resolving deltas: 100% (100/100), done.


### 03 Switch to the project directory

In [3]:
import os

# Move into your specific project folder on the remote machine
os.chdir("/content/CS224N-NLP-with-DL-2024/02-assignments/2024/assignment-3-v2")

# Print the directory contents to verify your python modules (.py files) are there
print("Current Working Directory:", os.getcwd())
print("\n=== Available Project Files ===")
!ls -la

Current Working Directory: /content/CS224N-NLP-with-DL-2024/02-assignments/2024/assignment-3

=== Available Project Files ===
total 66980
drwxr-xr-x 6 root root     4096 Aug 11 08:54 .
drwxr-xr-x 3 root root     4096 Aug 11 08:54 ..
-rw-r--r-- 1 root root    17076 Aug 11 08:54 01_implementation_v1.ipynb
-rw-r--r-- 1 root root     9014 Aug 11 08:54 02_training_local.ipynb
-rw-r--r-- 1 root root     9813 Aug 11 08:54 03_experiments_colab_v1.ipynb
-rw-r--r-- 1 root root  1235360 Aug 11 08:54 a3_spr24_student_handout.pdf
-rw-r--r-- 1 root root     1299 Aug 11 08:54 beam_search_diagnostics.py
-rw-r--r-- 1 root root      111 Aug 11 08:54 collect_submission.bat
-rw-r--r-- 1 root root       99 Aug 11 08:54 collect_submission.sh
-rw-r--r-- 1 root root      124 Aug 11 08:54 env-cpu.yml
-rw-r--r-- 1 root root      158 Aug 11 08:54 env-gpu.yml
-rw-r--r-- 1 root root        0 Aug 11 08:54 __init__.py
-rw-r--r-- 1 root root     2324 Aug 11 08:54 model_embeddings.py
-rw-r--r-- 1 root root    31954 Au

### 04 Import libraries

In [4]:
%load_ext autoreload
%autoreload 2

import torch

# Set up logging format and level
import logging
# logging.basicConfig(format="%(asctime)s - %(name)s - %(levelname)s - %(message)s")
logging.basicConfig(format="%(levelname)s:%(name)s:  %(message)s")

### 05 Set logging levels [OPTIONAL]

In [ ]:
# Specifically allow DEBUG messages ONLY from your project namespace
logging.getLogger("nmt").setLevel(logging.DEBUG)

In [ ]:
# Specifically disallow DEBUG messages ONLY from your project namespace
logging.getLogger("nmt").setLevel(logging.INFO)

### 06 Check hardware specifications [OPTIONAL]

In [5]:
# Check VM OS, RAM, and available disk space
print("=== Operating System ===")
!lsb_release -a

print("\n=== CPU Specifications ===")
!lscpu | grep "Model name\|CPU(s):"

print("\n=== System RAM ===")
!free -h

print("\n=== Disk Space ===")
!df -h /

=== Operating System ===
No LSB modules are available.
Distributor ID:	Ubuntu
Description:	Ubuntu 22.04.5 LTS
Release:	22.04
Codename:	jammy

=== CPU Specifications ===
CPU(s):                                  2
Model name:                              Intel(R) Xeon(R) CPU @ 2.00GHz
NUMA node0 CPU(s):                       0,1

=== System RAM ===
               total        used        free      shared  buff/cache   available
Mem:            12Gi       996Mi       6.9Gi       2.0Mi       4.8Gi        11Gi
Swap:             0B          0B          0B

=== Disk Space ===
Filesystem      Size  Used Avail Use% Mounted on
overlay         113G   47G   66G  42% /


### 07 Verify GPU Availability [OPTIONAL]

In [6]:
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU Device Name:", torch.cuda.get_device_name(0))
    print("CUDA Capability:", torch.cuda.get_device_capability(0))
else:
    print("Running on CPU.")

PyTorch Version: 2.11.0+cu128
CUDA Available: True
GPU Device Name: Tesla T4
CUDA Capability: (7, 5)


### 08 End the session [OPTIONAL]

In [36]:
from google.colab import runtime
runtime.unassign()

### 09 Pull the latest changes from the repository [OPTIONAL]

In [21]:
!git pull

remote: Enumerating objects: 12, done.
remote: Counting objects: 100% (12/12), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 7 (delta 3), reused 7 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (7/7), 10.31 KiB | 2.58 MiB/s, done.
From https://github.com/ilyarudyak/CS224N-NLP-with-DL-2024
   d2243a0..cb653f7  main       -> origin/main
Updating d2243a0..cb653f7
Fast-forward
 .../assignment-3/03_experiments_colab_v1.ipynb     | 2222 ++++++--------------
 .../2024/assignment-3/download_artifacts.py        |   48 +
 2 files changed, 711 insertions(+), 1559 deletions(-)
 create mode 100644 02-assignments/2024/assignment-3/download_artifacts.py


# 1 The First Run on a Toy Dataset

## 01 Run on a full dataset for 1 epoch on A100

In [11]:
!bash run_epoch_1.sh train_a100

## 03 Run on a full dataset for 30 epochs

In [16]:
!bash run.sh train_a100

In [17]:
!bash run.sh dev

In [18]:
!bash run.sh test